# Gesture Detection CNN — OpenMV RT1062

**EE6008 Deep Learning at the Edge | Group 4**

Trains a lightweight INT8 CNN for 5-class hand gesture recognition and exports a deployment-ready `.tflite` file for the OpenMV Cam RT1062.

| Property | Value |
|---|---|
| Dataset | [ryanbijujoseph/hand-gesture-dataset](https://www.kaggle.com/datasets/ryanbijujoseph/hand-gesture-dataset) |
| Classes | backward, forward, left, right, unknown |
| Split | 70% train / 15% val / 15% test |
| Input | 48×48 grayscale |
| Architecture | Conv(32) → Conv(64) → Conv(128) → GAP → Dropout → Dense(5) |
| Export | INT8 TFLite, TFL2 schema (OpenMV compatible) |
| Target device | OpenMV RT1062 — Cortex-M7, 8MB SRAM |

## 1 — Imports

In [ ]:
import os, shutil, pathlib, zipfile, glob
import numpy as np
import matplotlib.pyplot as plt
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

import tensorflow as tf
from tensorflow.keras import layers, callbacks

print(f"TensorFlow : {tf.__version__}")
print(f"NumPy      : {np.__version__}")

## 2 — Kaggle Dataset Download

Dataset: **[ryanbijujoseph/hand-gesture-dataset](https://www.kaggle.com/datasets/ryanbijujoseph/hand-gesture-dataset)**

Credentials are written directly from the hardcoded API key — no file upload required. Just run the cell.

In [ ]:
import subprocess, sys, os, json, pathlib

# Install Kaggle API
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "kaggle"], check=True)

# Write credentials directly — no file upload needed
os.makedirs("/root/.kaggle", exist_ok=True)
with open("/root/.kaggle/kaggle.json", "w") as f:
    json.dump({"username": "crosskun", "key": "KGAT_fb3c55a20f220b904397e21a27488872"}, f)
os.chmod("/root/.kaggle/kaggle.json", 0o600)

print("Kaggle credentials written.")

In [ ]:
import kaggle

DOWNLOAD_DIR = pathlib.Path("kaggle_data")
DOWNLOAD_DIR.mkdir(exist_ok=True)

DATASET_SLUG = "ryanbijujoseph/hand-gesture-dataset"

print(f"Downloading: {DATASET_SLUG} ...")
kaggle.api.authenticate()
kaggle.api.dataset_download_files(
    DATASET_SLUG,
    path=str(DOWNLOAD_DIR),
    unzip=True,
    quiet=False,
)
print("Download complete.")

# ── Auto-detect the class folder root ────────────────────────────────────────
# Walk until we find a directory that contains all 5 class subfolders
CLASSES_SET = {"forward", "backward", "left", "right", "unknown"}
DATASET_DIR = None

for root, dirs, files in os.walk(DOWNLOAD_DIR):
    if CLASSES_SET.issubset({d.lower() for d in dirs}):
        DATASET_DIR = pathlib.Path(root)
        break

if DATASET_DIR is None:
    raise RuntimeError(
        f"Could not find the 5 class folders inside {DOWNLOAD_DIR}.\n"
        f"Contents: {list(DOWNLOAD_DIR.rglob('*'))[:30]}"
    )

print(f"\nDataset root : {DATASET_DIR}")
print("\nClass counts:")
total = 0
for cls in sorted(CLASSES_SET):
    cls_dir = DATASET_DIR / cls
    # match any case
    if not cls_dir.exists():
        cls_dir = next((DATASET_DIR / d for d in os.listdir(DATASET_DIR)
                        if d.lower() == cls), None)
    n = len(list(cls_dir.glob("*"))) if cls_dir and cls_dir.exists() else 0
    total += n
    print(f"  {cls:<12} {n:>5} images")
print(f"  {'TOTAL':<12} {total:>5} images")

## 3 — Constants

- **48×48 grayscale** input — 8× smaller tensor than 96×96 RGB565, confirmed to fit the RT1062 tensor arena.
- `DATASET_DIR` is set automatically from the Kaggle download above.
- **Split:** 70% train / 15% val / 15% test — stratified by class folder.
- Set `EPOCHS = 45` with early stopping; training typically converges by epoch 15–20.

In [ ]:
IMG_H, IMG_W = 48, 48
CLASSES      = ["forward", "backward", "left", "right", "unknown"]
NUM_CLASSES  = len(CLASSES)
BATCH_SIZE   = 32
EPOCHS       = 45

# DATASET_DIR is set by the Kaggle download cell above.
# To use a local folder instead, uncomment and edit the line below:
# DATASET_DIR = pathlib.Path("/path/to/your/data")

MODEL_DIR = pathlib.Path("model_output")
MODEL_DIR.mkdir(exist_ok=True)

print(f"Dataset : {DATASET_DIR}")
print(f"Output  : {MODEL_DIR.resolve()}")
print(f"Classes : {CLASSES}")

## 4 — Dataset — Train / Val / Test

**70 / 15 / 15 split** — all three sets are stratified by class folder.

- `train_ds` (70%) — used for gradient updates
- `val_ds` (15%) — used during training to monitor overfitting and drive callbacks
- `test_ds` (15%) — held out completely until final evaluation after training

Pixels are normalised to **[0, 1]** float32. The model contains no internal `Rescaling` layer — having normalisation in only one place prevents accidental double-scaling.

In [ ]:
norm = lambda x, y: (tf.cast(x, tf.float32) / 255.0, y)

common = dict(
    labels       = "inferred",
    label_mode   = "categorical",
    class_names  = CLASSES,
    color_mode   = "grayscale",
    batch_size   = BATCH_SIZE,
    image_size   = (IMG_H, IMG_W),
    seed         = 42,
)

# ── Train: 70% ───────────────────────────────────────────────────────────────
train_ds = tf.keras.utils.image_dataset_from_directory(
    DATASET_DIR, subset="training", validation_split=0.30, **common
).map(norm).prefetch(tf.data.AUTOTUNE)

# ── Remaining 30% → split evenly into val (15%) and test (15%) ───────────────
remaining_ds = tf.keras.utils.image_dataset_from_directory(
    DATASET_DIR, subset="validation", validation_split=0.30, **common
).map(norm)

n_remaining  = remaining_ds.cardinality().numpy()   # number of batches
n_val        = n_remaining // 2

val_ds  = remaining_ds.take(n_val).prefetch(tf.data.AUTOTUNE)
test_ds = remaining_ds.skip(n_val).prefetch(tf.data.AUTOTUNE)

# ── Summary ───────────────────────────────────────────────────────────────────
n_train = train_ds.cardinality().numpy()
n_test  = test_ds.cardinality().numpy()
print(f"Train batches : {n_train}  (~{n_train * BATCH_SIZE} images)")
print(f"Val   batches : {n_val}   (~{n_val   * BATCH_SIZE} images)")
print(f"Test  batches : {n_test}   (~{n_test  * BATCH_SIZE} images)")

## 5 — Model Architecture

```
Input (48×48×1)
  │
Conv2D(32, 3×3, relu)  → MaxPool(2×2)  →  24×24×32
  │
Conv2D(64, 3×3, relu)  → MaxPool(2×2)  →  12×12×64
  │
Conv2D(128, 3×3, relu) → MaxPool(2×2)  →   6×6×128
  │
GlobalAveragePooling2D                 →       128
  │
Dropout(0.3)
  │
Dense(5, softmax)
```

**Why GAP instead of Flatten?**  
`Flatten` on 6×6×128 produces a 4,608-element vector — this single tensor overflows the RT1062 arena.  
`GlobalAveragePooling2D` collapses it to 128 elements, an **8× reduction**, keeping peak tensor memory at ~4.5 KB.

In [ ]:
def build_model():
    inp = tf.keras.Input(shape=(IMG_H, IMG_W, 1), name="input_image")

    x = layers.Conv2D(32,  3, padding="same", activation="relu")(inp)
    x = layers.MaxPooling2D(2, 2)(x)            # -> 24×24×32

    x = layers.Conv2D(64,  3, padding="same", activation="relu")(x)
    x = layers.MaxPooling2D(2, 2)(x)            # -> 12×12×64

    x = layers.Conv2D(128, 3, padding="same", activation="relu")(x)
    x = layers.MaxPooling2D(2, 2)(x)            # ->  6×6×128

    x = layers.GlobalAveragePooling2D()(x)      # ->     128
    x = layers.Dropout(0.3)(x)
    out = layers.Dense(NUM_CLASSES, activation="softmax", name="predictions")(x)

    return tf.keras.Model(inp, out, name="gesture_cnn")


model = build_model()
model.summary()

## 6 — Training

- **Optimiser:** Adam, lr=1e-3
- **Loss:** categorical cross-entropy
- **EarlyStopping:** patience=8, monitors `val_accuracy`, restores best weights
- **ReduceLROnPlateau:** halves LR after 4 stagnant epochs, floor at 1e-6
- **ModelCheckpoint:** saves best epoch to `model_output/best_gesture_cnn.keras`

In [ ]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss="categorical_crossentropy",
    metrics=["accuracy"],
)

cbs = [
    callbacks.EarlyStopping(
        monitor="val_accuracy", patience=8, restore_best_weights=True),
    callbacks.ReduceLROnPlateau(
        monitor="val_loss", factor=0.5, patience=4, min_lr=1e-6),
    callbacks.ModelCheckpoint(
        str(MODEL_DIR / "best_gesture_cnn.keras"),
        monitor="val_accuracy", save_best_only=True),
]

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=cbs,
    verbose=1,
)

### Training Curves

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(history.history["accuracy"],     label="train")
ax1.plot(history.history["val_accuracy"], label="val")
ax1.set_title("Accuracy")
ax1.set_xlabel("Epoch")
ax1.legend()
ax1.grid(True)

ax2.plot(history.history["loss"],     label="train")
ax2.plot(history.history["val_loss"], label="val")
ax2.set_title("Loss")
ax2.set_xlabel("Epoch")
ax2.legend()
ax2.grid(True)

plt.tight_layout()
plt.savefig(str(MODEL_DIR / "training_curves.png"), dpi=150)
plt.show()

best_val_acc = max(history.history["val_accuracy"])
print(f"Best val_accuracy : {best_val_acc:.4f}  ({best_val_acc*100:.2f}%)")

### Test Set Evaluation

Evaluated on the **held-out test set** (15% of data, never seen during training or validation).

In [ ]:
test_loss, test_acc = model.evaluate(test_ds, verbose=1)
print(f"\nTest loss     : {test_loss:.4f}")
print(f"Test accuracy : {test_acc:.4f}  ({test_acc*100:.2f}%)")

### Confusion Matrix

In [ ]:
y_true, y_pred = [], []

for x_batch, y_batch in test_ds:
    preds = model.predict(x_batch, verbose=0)
    y_true.extend(np.argmax(y_batch.numpy(), axis=1))
    y_pred.extend(np.argmax(preds,           axis=1))

y_true = np.array(y_true)
y_pred = np.array(y_pred)

# Compute confusion matrix manually (no sklearn required)
cm = np.zeros((NUM_CLASSES, NUM_CLASSES), dtype=int)
for t, p in zip(y_true, y_pred):
    cm[t, p] += 1

fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(cm, cmap="Blues")
ax.set_xticks(range(NUM_CLASSES)); ax.set_xticklabels(CLASSES, rotation=45, ha="right")
ax.set_yticks(range(NUM_CLASSES)); ax.set_yticklabels(CLASSES)
ax.set_xlabel("Predicted"); ax.set_ylabel("True")
ax.set_title("Confusion Matrix — Test Set")
plt.colorbar(im, ax=ax)
for i in range(NUM_CLASSES):
    for j in range(NUM_CLASSES):
        ax.text(j, i, cm[i, j], ha="center", va="center",
                color="white" if cm[i, j] > cm.max() / 2 else "black")
plt.tight_layout()
plt.savefig(str(MODEL_DIR / "confusion_matrix.png"), dpi=150)
plt.show()

## 7 — Export: INT8 TFLite

Full-integer quantisation — weights, activations, **and** input/output tensors are all INT8.  
This is mandatory for OpenMV's `ml` module which does not support FP32 inference.

A **representative dataset** (200 real training images) is required so the converter can calibrate activation ranges — without it the output/input tensors remain FP32 regardless of the `inference_input_type` setting.

> `model.export()` is the Keras 3 API for producing a SavedModel. The older `model.save('path/')` raises a `ValueError` in Keras 3+.

In [ ]:
saved_model_path = str(MODEL_DIR / "gesture_cnn_savedmodel")
if pathlib.Path(saved_model_path).exists():
    shutil.rmtree(saved_model_path)
model.export(saved_model_path)

def representative_dataset():
    """200 samples from the training set — float32 [0,1]."""
    count = 0
    for batch_x, _ in train_ds:
        for img in batch_x:
            if count >= 200:
                return
            yield [tf.expand_dims(img, 0)]   # shape [1, 48, 48, 1]
            count += 1

converter = tf.lite.TFLiteConverter.from_saved_model(saved_model_path)
converter.optimizations              = [tf.lite.Optimize.DEFAULT]
converter.representative_dataset    = representative_dataset
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter.inference_input_type      = tf.int8
converter.inference_output_type     = tf.int8

tflite_model = converter.convert()

tflite_path = MODEL_DIR / "gesture_cnn_int8.tflite"
tflite_path.write_bytes(tflite_model)
print(f"[OK] TFLite written: {tflite_path}  ({len(tflite_model)/1024:.1f} KB)")

## 8 — Schema Patch: TFL3 → TFL2

TF 2.x exports flatbuffers with the file identifier `TFL3`. The OpenMV v4.8.1 firmware `ml` module only accepts `TFL2`.

**Flatbuffer binary layout:**
```
bytes[0:4]  root table offset  (uint32 LE)
bytes[4:8]  file identifier    b"TFL3"  ← byte 7 = 0x33
bytes[8:]   model weights + graph        (untouched)
```

Flipping byte 7 from `0x33` → `0x32` downgrades the header. Weights and graph structure are completely untouched.

In [ ]:
data       = bytearray(tflite_path.read_bytes())
identifier = bytes(data[4:8])
print(f"File identifier before patch: {identifier}")

assert identifier == b"TFL3", f"Unexpected identifier {identifier!r} — check TF version"

data[7]      = ord("2")
patched_path = MODEL_DIR / "gesture_cnn_int8_tfl2.tflite"
patched_path.write_bytes(bytes(data))

print(f"File identifier after  patch: {bytes(data[4:8])}")
print(f"[OK] Patched file: {patched_path}  ({len(data)/1024:.1f} KB)")

## 9 — Validation

Runs `allocate_tensors()` and a dummy forward pass on the **TFL3** file (the host TF interpreter rejects TFL2 — that header is for OpenMV only). Tensor shapes and quantisation params are identical between both files since only byte 7 differs.

This catches any tensor allocation or shape errors before the file is deployed to the MCU.

In [ ]:
interp = tf.lite.Interpreter(model_path=str(tflite_path))   # validate on TFL3 (host)
interp.allocate_tensors()

inp_det = interp.get_input_details()[0]
out_det = interp.get_output_details()[0]

print(f"Input  : shape={inp_det['shape']}  dtype={inp_det['dtype']}")
print(f"Output : shape={out_det['shape']}  dtype={out_det['dtype']}")
print(f"Input  quant : scale={inp_det['quantization'][0]:.6f},  zero_point={inp_det['quantization'][1]}")
print(f"Output quant : scale={out_det['quantization'][0]:.6f},  zero_point={out_det['quantization'][1]}")

# Dummy forward pass
dummy = np.zeros(inp_det["shape"], dtype=inp_det["dtype"])
interp.set_tensor(inp_det["index"], dummy)
interp.invoke()
output = interp.get_tensor(out_det["index"])
print(f"Dummy output (int8): {output}")

# Tensor arena estimate
arena_bytes = sum(
    int(np.prod(d["shape"])) * np.dtype(d["dtype"]).itemsize
    for d in interp.get_tensor_details() if len(d["shape"]) > 0
)
print(f"\n[OK] No tensor errors.")
print(f"Estimated tensor arena : {arena_bytes/1024:.1f} KB")
print(f"RT1062 SRAM            : 8192 KB")
print(f"Headroom               : {(8192*1024 - arena_bytes)/1024:.0f} KB")

## 10 — Output Summary

In [ ]:
print("=" * 60)
print("RESULTS")
print("=" * 60)
print(f"Best val  accuracy : {best_val_acc*100:.2f}%")
print(f"Test      accuracy : {test_acc*100:.2f}%")
print(f"Test      loss     : {test_loss:.4f}")
print()
print("OUTPUT FILES")
print("-" * 60)
for f in sorted(MODEL_DIR.rglob("*")):
    if f.is_file():
        print(f"  {f.stat().st_size/1024:7.1f} KB   {f.relative_to(MODEL_DIR)}")
print()
print("DEPLOY TO OPENMV")
print("-" * 60)
print(f"  Copy '{patched_path.name}' to /flash/ on the OpenMV device")
print()
print("OPENMV INPUT QUANTISATION")
print("-" * 60)
print(f"  scale      : {inp_det['quantization'][0]:.6f}")
print(f"  zero_point : {inp_det['quantization'][1]}")
print("  Formula    : int8 = uint8_pixel - 128  (simplified, valid when scale ~= 1/255)")
print("=" * 60)